In [1]:


import pandas as pd
path_data_dir = "../../data/inputs/" 
train_data_file = path_data_dir + "hoteles-entrena.csv" 
test_data_file = path_data_dir + "hoteles-prueba.csv"
#train data reanding
train_data = pd.read_csv(train_data_file, sep=",")
print('')
print('train data info:')
train_data.info()
test_data = pd.read_csv(test_data_file, sep=",")
print('')
print('test data info:')
test_data.info()


train data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52981 entries, 0 to 52980
Data columns (total 25 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           52981 non-null  object 
 1   lead_time                       52981 non-null  int64  
 2   stays_in_weekend_nights         52981 non-null  int64  
 3   stays_in_week_nights            52981 non-null  int64  
 4   adults                          52981 non-null  int64  
 5   children                        52981 non-null  object 
 6   meal                            52981 non-null  object 
 7   country                         52681 non-null  object 
 8   market_segment                  52981 non-null  object 
 9   distribution_channel            52981 non-null  object 
 10  is_repeated_guest               52981 non-null  int64  
 11  previous_cancellations          52981 non-null  int64  
 12  previous_booki

In [2]:
train_data.head(3)  

,hotel,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,meal,country,market_segment,distribution_channel,...,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,average_daily_rate,required_car_parking_spaces,total_of_special_requests,arrival_date
0,Resort_Hotel,342,0,0,2,none,BB,PRT,Direct,Direct,...,3,No_Deposit,NaN,NaN,0,Transient,0.0,none,0,2015-07-01
1,Resort_Hotel,737,0,0,2,none,BB,PRT,Direct,Direct,...,4,No_Deposit,NaN,NaN,0,Transient,0.0,none,0,2015-07-01
2,Resort_Hotel,7,0,1,1,none,BB,GBR,Direct,Direct,...,0,No_Deposit,NaN,NaN,0,Transient,75.0,none,0,2015-07-01


In [3]:

# Count unique values in 'meal' and put them into a DataFrame
train_meal_counts = train_data['meal'].value_counts().reset_index()
train_meal_counts['%'] = train_meal_counts['count']/sum(train_meal_counts['count'])
train_meal_counts.columns = ['meal', 'count', '%']
print(train_meal_counts)
print('')
# Count unique values in 'meal' and put them into a DataFrame
test_meal_counts = test_data['meal'].value_counts().reset_index()
test_meal_counts['%'] = test_meal_counts['count']/sum(test_meal_counts['count'])
test_meal_counts.columns = ['meal', 'count', '%']
print(test_meal_counts)


        meal  count         %
0         BB  40865  0.771314
1         HB   6624  0.125026
2         SC   4637  0.087522
3  Undefined    632  0.011929
4         FB    223  0.004209

        meal  count         %
0         BB  16935  0.763354
1         HB   2855  0.128691
2         SC   2047  0.092270
3  Undefined    251  0.011314
4         FB     97  0.004372


In [4]:
# Check if 'meal' is 'SC' or 'Undefined' and if there are children (children != 'none')
mask = train_data['meal'].isin(['SC', 'Undefined']) & (train_data['children'] != 'none')
coincidence_count = mask.sum()
total_sc_undefined = train_data['meal'].isin(['SC', 'Undefined']).sum()
print(f"Number of cases where meal is SC or Undefined and there are children: {coincidence_count}")
print(f"Total cases with meal SC or Undefined: {total_sc_undefined}")
print(f"Proportion: {coincidence_count / total_sc_undefined:.4f}")
# Check coincidence for each meal type: BB, HB, FB
for meal_type in ['BB', 'HB', 'FB']:
    mask_meal = (train_data['meal'] == meal_type) & (train_data['children'] != 'none')
    coincidence = mask_meal.sum()
    total_meal = (train_data['meal'] == meal_type).sum()
    proportion = coincidence / total_meal if total_meal > 0 else 0
    print(f"Meal: {meal_type}")
    print(f"  Number of cases with children: {coincidence}")
    print(f"  Total cases: {total_meal}")
    print(f"  Proportion: {proportion:.4f}")


Number of cases where meal is SC or Undefined and there are children: 146
Total cases with meal SC or Undefined: 5269
Proportion: 0.0277
Meal: BB
  Number of cases with children: 3480
  Total cases: 40865
  Proportion: 0.0852
Meal: HB
  Number of cases with children: 665
  Total cases: 6624
  Proportion: 0.1004
Meal: FB
  Number of cases with children: 43
  Total cases: 223
  Proportion: 0.1928


In [5]:
# Extract weeks in year and time span from stays_in_week_nights and stays_in_weekends_nights

# For train_data
train_data['total_nights'] = train_data['stays_in_week_nights'] + train_data['stays_in_weekend_nights']
train_data['weeks_in_year'] = ((train_data['total_nights'] / 7).round().clip(lower=1, upper=52)).astype(int)
train_data['time_span'] = train_data['total_nights']
train_data['arrival_week'] = pd.to_datetime(train_data['arrival_date']).dt.isocalendar().week
train_data['arrival_week'] = train_data['arrival_week'].astype('int64')

# For test_data
test_data['total_nights'] = test_data['stays_in_week_nights'] + test_data['stays_in_weekend_nights']
test_data['weeks_in_year'] = ((test_data['total_nights'] / 7).round().clip(lower=1, upper=52)).astype(int)
test_data['time_span'] = test_data['total_nights']
test_data['arrival_week'] = pd.to_datetime(test_data['arrival_date']).dt.isocalendar().week
test_data['arrival_week'] = test_data['arrival_week'].astype('int64')

In [6]:
train_data.loc[train_data['meal'] == 'Undefined', 'meal'] = 'SC'
test_data.loc[test_data['meal'] == 'Undefined', 'meal'] = 'SC'
# Count unique values in 'meal' and put them into a DataFrame
train_meal_counts = train_data['meal'].value_counts().reset_index()
train_meal_counts['%'] = train_meal_counts['count']/sum(train_meal_counts['count'])
train_meal_counts.columns = ['meal', 'count', '%']
print(train_meal_counts)
print('')
# Count unique values in 'meal' and put them into a DataFrame
test_meal_counts = test_data['meal'].value_counts().reset_index()
test_meal_counts['%'] = test_meal_counts['count']/sum(test_meal_counts['count'])
test_meal_counts.columns = ['meal', 'count', '%']
print(test_meal_counts)


  meal  count         %
0   BB  40865  0.771314
1   HB   6624  0.125026
2   SC   5269  0.099451
3   FB    223  0.004209

  meal  count         %
0   BB  16935  0.763354
1   HB   2855  0.128691
2   SC   2298  0.103584
3   FB     97  0.004372


In [7]:
from dateutil.easter import easter

# Add "school_vacations" column: 1 if arrival_date is in December, June, July, August, or Easter week

def is_school_vacation(date_str):
    date = pd.to_datetime(date_str, errors='coerce')
    if pd.isna(date):
        return 0
    # Months: December (12), June (6), July (7), August (8)
    if date.month in [6, 7, 8, 12]:
        return 1
    # Easter: calculate Easter Sunday and check if date is in Easter week (Sunday to Saturday)
    easter_sunday = easter(date.year)
    easter_week = pd.date_range(easter_sunday - pd.Timedelta(days=easter_sunday.weekday()+1), periods=7)
    if date in easter_week:
        return 1
    return 0

train_data['school_vacations'] = train_data['arrival_date'].apply(is_school_vacation)
test_data['school_vacations'] = test_data['arrival_date'].apply(is_school_vacation)

In [8]:
print(train_data.info())
print('')
print(test_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52981 entries, 0 to 52980
Data columns (total 30 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           52981 non-null  object 
 1   lead_time                       52981 non-null  int64  
 2   stays_in_weekend_nights         52981 non-null  int64  
 3   stays_in_week_nights            52981 non-null  int64  
 4   adults                          52981 non-null  int64  
 5   children                        52981 non-null  object 
 6   meal                            52981 non-null  object 
 7   country                         52681 non-null  object 
 8   market_segment                  52981 non-null  object 
 9   distribution_channel            52981 non-null  object 
 10  is_repeated_guest               52981 non-null  int64  
 11  previous_cancellations          52981 non-null  int64  
 12  previous_bookings_not_canceled  

In [9]:
train_data['children'] = train_data['children'].apply(lambda x: 0 if x == 'none' else 1)
#type(train_data['children'])

In [10]:
missing_in_test = set(train_data.columns) - set(test_data.columns)
print("Columns in train_data but not in test_data:", missing_in_test)

Columns in train_data but not in test_data: {'children'}


In [11]:
# Convert children to numeric
train_data['children'] = pd.to_numeric(train_data['children'], errors='coerce')

# Define target: presence of children (0 or 1)
train_data['has_children'] = (train_data['children'] > 0).astype(int)
y = train_data['has_children']
train_data['children'].isna().sum()

0

Create a Pipeline for preprocessing and modeling

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import os

# Define features (exclude 'children')
X = train_data.drop(columns=['children', 'has_children'])

# Split categorical/numerical
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(exclude=['object']).columns.tolist()

# Imputers (no data evalable set median value for numerical, most_frequent for categorical)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
 
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Model
clf_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train/test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit
clf_rf.fit(X_train, y_train)
# add the new accuracy value to the csv, including the number of validation samples
val_acc_file = '../../data/outputs/val_acc.csv'
val_acc = clf_rf.score(X_val, y_val)

if os.path.exists(val_acc_file):
    df_acc = pd.read_csv(val_acc_file)
    # Get the number of validation samples
    val_n = df_acc['val_n'].iloc[-1]
    # Append new row with correct val_n
    method = 'RandomForestClassifier'
    df_acc = pd.concat([df_acc, pd.DataFrame([{'val_acc': val_acc, 'val_n': val_n+1, 'method':  method}])], ignore_index=True)
    df_acc = pd.concat([df_acc, pd.DataFrame([{'val_acc': val_acc, 'val_n': val_n+1}])], ignore_index=True)

else:
    df_acc = pd.DataFrame({'val_acc': [val_acc], 'val_n': 1})

df_acc.to_csv(val_acc_file, index=False)

clf_rf.fit(X_train, y_train)
y_pred = clf_rf.predict(X_val)
print("Accuracy:", clf_rf.score(X_val, y_val))


Accuracy: 0.9442294989147872


In [16]:
#  prep test data for pipeline}
df_test = test_data.copy()




In [17]:

# Predict on validation set
y_pred = clf_rf.predict(df_test)




In [18]:
# Predict probabilities for validation set (y=1 means has children)
y_proba = clf_rf.predict_proba(df_test)[:, 1]
X_test = df_test.copy()
X_test['id'] = range(1, len(df_test) + 1)

In [19]:
# Create a  DataFrame for submissio 
df_submission = pd.DataFrame({
    'id': X_test['id'],
    'prob': y_proba
})
df_submission.head()

,id,prob
0,1,0.03
1,2,0.07
2,3,0.54
3,4,0.06
4,5,0.01


In [20]:
from datetime import datetime
date = datetime.now().strftime('%y%m%d')  # format yy_mm_dd
new_sumbission_file = f'../../data/outputs/submission_rdf_{date}_rf.csv'

df_submission.to_csv(new_sumbission_file, index=False)
print(f'sumbission saved to: {new_sumbission_file}')

sumbission saved to: ../../data/outputs/submission_rdf_251004_rf.csv
